## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip3 install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.13 requires numpy<2,>=1, but you have numpy 2.3.3 which is incompatible.
langchain 0.1.1 requires numpy<2,>=1, but you have numpy 2.3.3 which is incompatible.


In [2]:
# For installing the libraries & downloading models from HF Hub
!pip3 install huggingface_hub==0.23.2 pandas==1.5.3 tiktoken==0.6.0 pymupdf==1.25.1 langchain==0.1.1 langchain-community==0.0.13 chromadb==1.1.0 sentence-transformers==2.3.1 numpy==1.26.4 -q

In [3]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Question Answering using LLM

#### Downloading and Loading the model

In [4]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [5]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

In [6]:
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
)

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf (version unknown)
llama_model_loader: - tensor    0:                token_embd.weight q6_K     [  4096, 32000,     1,     1 ]
llama_model_loader: - tensor    1:              blk.0.attn_q.weight q6_K     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    2:              blk.0.attn_k.weight q6_K     [  4096,  1024,     1,     1 ]
llama_model_loader: - tensor    3:              blk.0.attn_v.weight q6_K     [  4096,  1024,     1,     1 ]
llama_model_loader: - tensor    4:         blk.0.attn_output.weight q6_K     [  4096,  4096,     1,     1 ]
llama_model_loader: - tensor    5:            blk.0.ffn_gate.weight q6_K     [  4096, 14336,     1,     1 ]
llama_model_loader: - tensor    6:              blk.0.ffn_up.weight q6_K     [  4096, 143

#### Response

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [8]:
input1 = "What is the protocol for managing sepsis in a critical care unit?"
response(input1)


llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.62 ms /   128 runs   (    0.42 ms per token,  2387.21 tokens per second)
llama_print_timings: prompt eval time =   805.62 ms /    16 tokens (   50.35 ms per token,    19.86 tokens per second)
llama_print_timings:        eval time = 20973.44 ms /   127 runs   (  165.15 ms per token,     6.06 tokens per second)
llama_print_timings:       total time = 22007.45 ms


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. Resusc'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [9]:
input2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(input2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.87 ms /   128 runs   (    0.42 ms per token,  2376.13 tokens per second)
llama_print_timings: prompt eval time =  1308.49 ms /    32 tokens (   40.89 ms per token,    24.46 tokens per second)
llama_print_timings:        eval time = 20837.71 ms /   127 runs   (  164.08 ms per token,     6.09 tokens per second)
llama_print_timings:       total time = 22374.11 ms


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The following are common symptoms of appendicitis:\n\n1. Abdominal pain: The pain typically starts around the navel area and then shifts to the lower right side of the abdomen. It may be constant or come and go, and it can range from mild to severe.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and discomfort.\n3. Nausea and vomiting:'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [10]:
input3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(input3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.81 ms /   128 runs   (    0.42 ms per token,  2378.65 tokens per second)
llama_print_timings: prompt eval time =  1408.54 ms /    34 tokens (   41.43 ms per token,    24.14 tokens per second)
llama_print_timings:        eval time = 20813.07 ms /   127 runs   (  163.88 ms per token,     6.10 tokens per second)
llama_print_timings:       total time = 22449.04 ms


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [11]:
input4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(input4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.76 ms /   128 runs   (    0.42 ms per token,  2381.00 tokens per second)
llama_print_timings: prompt eval time =  1168.94 ms /    28 tokens (   41.75 ms per token,    23.95 tokens per second)
llama_print_timings:        eval time = 20816.01 ms /   127 runs   (  163.91 ms per token,     6.10 tokens per second)
llama_print_timings:       total time = 22214.46 ms


"\n\nA person who has sustained a physical injury to the brain, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for individuals with TBIs:\n\n1. Emergency care: In case of a severe TBI, the first priority is to ensure the person's safety and provide emergency medical care. This may include administering oxygen, controlling bleeding, preventing further injury, and monitoring vital signs.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various"

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [12]:
input5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(input5)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    55.84 ms /   128 runs   (    0.44 ms per token,  2292.47 tokens per second)
llama_print_timings: prompt eval time =  1442.69 ms /    35 tokens (   41.22 ms per token,    24.26 tokens per second)
llama_print_timings:        eval time = 20868.34 ms /   127 runs   (  164.32 ms per token,     6.09 tokens per second)
llama_print_timings:       total time = 22547.41 ms


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sl"

## Question Answering using LLM with Prompt Engineering

In [13]:
system_prompt = """
You are an AI assistant with expertise in the medical domain.
Your role is to provide clear, concise, and relevant information based on established medical guidelines and reputable health sources such as WHO, CDC, and Mayo Clinic.
Responses must adhere to professional medical standards and be suitable for review by qualified healthcare providers.
Do not provide personal medical advice, diagnosis, or treatment recommendations.
If the information is unavailable or uncertain, explicitly state that you do not know and recommend consulting a licensed healthcare professional.
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [14]:
prompt_input1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"
response(prompt_input1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    55.31 ms /   128 runs   (    0.43 ms per token,  2314.35 tokens per second)
llama_print_timings: prompt eval time =  5088.13 ms /   129 tokens (   39.44 ms per token,    25.35 tokens per second)
llama_print_timings:        eval time = 20942.07 ms /   127 runs   (  164.90 ms per token,     6.06 tokens per second)
llama_print_timings:       total time = 26268.83 ms


'\n\nSepsis is a life-threatening condition caused by a dysregulated response to infection. In a critical care unit, managing sepsis involves several key steps based on established guidelines from organizations such as the Surviving Sepsis Campaign (SSC) and the Society of Critical Care Medicine (SCCM). Here are some general principles for managing sepsis in a critical care unit:\n\n1. Early recognition and diagnosis: Recognize sepsis early by identifying signs and symptoms, such as fever or hypothermia, tachycardia or bradycardia, resp'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [15]:
prompt_input2 = system_prompt + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(prompt_input2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.76 ms /   128 runs   (    0.42 ms per token,  2380.78 tokens per second)
llama_print_timings: prompt eval time =  1321.00 ms /    32 tokens (   41.28 ms per token,    24.22 tokens per second)
llama_print_timings:        eval time = 20965.03 ms /   127 runs   (  165.08 ms per token,     6.06 tokens per second)
llama_print_timings:       total time = 22518.07 ms


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The most common symptoms of appendicitis include:\n1. Sudden pain in the lower right abdomen, often starting around the navel and then moving to the right side.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Fever, which may be low-grade at first but can rise as high as 101°F (38.3°C) or more.\n4. V'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [16]:
prompt_input3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(prompt_input3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.58 ms /   128 runs   (    0.42 ms per token,  2388.86 tokens per second)
llama_print_timings: prompt eval time =  1427.28 ms /    34 tokens (   41.98 ms per token,    23.82 tokens per second)
llama_print_timings:        eval time = 20912.35 ms /   127 runs   (  164.66 ms per token,     6.07 tokens per second)
llama_print_timings:       total time = 22571.88 ms


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the sudden loss of hair in small, round patches on the scalp or other areas of the body. The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system.\n\nThere is no cure for alopecia areata, but there are several treatments that can help stimulate hair regrowth and potentially prevent further hair loss:\n\n1. Corticosteroids: These medications can be"

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [17]:
prompt_input4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(prompt_input4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.73 ms /   128 runs   (    0.42 ms per token,  2382.10 tokens per second)
llama_print_timings: prompt eval time =  1175.37 ms /    28 tokens (   41.98 ms per token,    23.82 tokens per second)
llama_print_timings:        eval time = 20995.85 ms /   127 runs   (  165.32 ms per token,     6.05 tokens per second)
llama_print_timings:       total time = 22404.31 ms


'\n\nI cannot provide personal medical advice or recommendations for specific treatments as I do not have access to your individual medical history or current condition. However, I can share some general information about the treatment options for brain injuries based on established medical guidelines and reputable health sources such as the World Health Organization (WHO), Centers for Disease Control and Prevention (CDC), and Mayo Clinic.\n\nBrain injuries can range from mild to severe, and the treatment approach depends on the severity and location of the injury. Here are some common treatments for brain injuries:\n\n1. Emergency care: For severe'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [18]:
prompt_input5 = system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(prompt_input5)

Llama.generate: prefix-match hit


'\n\nA leg fracture, specifically a tibia or fibula fracture, is a common injury that can occur during hiking due to falls, tripping over rocks, or other traumatic events. Here are the necessary precautions and treatment steps for such an individual:\n\n1. Assess the situation: Ensure the safety of both the injured person and any companions. Check for signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If there is a risk of further injury or the person shows signs of shock, call emergency medical services immediately.\n\n2. Immobilize'


llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.97 ms /   128 runs   (    0.42 ms per token,  2371.64 tokens per second)
llama_print_timings: prompt eval time =  1455.47 ms /    35 tokens (   41.58 ms per token,    24.05 tokens per second)
llama_print_timings:        eval time = 20943.56 ms /   127 runs   (  164.91 ms per token,     6.06 tokens per second)
llama_print_timings:       total time = 22631.90 ms


### Output of queries with LLM parameter tuning combinations


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [19]:
prompt_input1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"
response(prompt_input1, temperature= 0.5, max_tokens=512, top_p=0.98, top_k= 10)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =   184.66 ms /   512 runs   (    0.36 ms per token,  2772.69 tokens per second)
llama_print_timings: prompt eval time =   599.39 ms /    14 tokens (   42.81 ms per token,    23.36 tokens per second)
llama_print_timings:        eval time = 84491.71 ms /   511 runs   (  165.35 ms per token,     6.05 tokens per second)
llama_print_timings:       total time = 86105.32 ms


"\n\nSepsis is a life-threatening condition caused by a dysregulated response to infection. In a critical care unit, managing sepsis involves a multidisciplinary approach aimed at addressing the underlying infection, supporting organ function, and preventing complications. Here are some key aspects of sepsis management in a critical care setting:\n\n1. Early recognition and diagnosis: Recognize sepsis early based on clinical signs and laboratory findings, such as fever or hypothermia, tachycardia or bradycardia, respiratory distress, altered mental status, lactic acidosis, and leukocytosis or leukopenia. Use validated scoring systems like the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score or the Quick Sequential [Sepsis-related] Organ Failure Assessment (qSOFA) to help identify patients at risk for sepsis.\n\n2. Source control: Identify and treat the source of infection as soon as possible, which may involve surgical interventions, drainage procedures, or antimicrobi

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [20]:
prompt_input2 = system_prompt + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(prompt_input2, temperature= 0.10, max_tokens=512, top_p=0.96, top_k= 15)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =   144.01 ms /   392 runs   (    0.37 ms per token,  2722.11 tokens per second)
llama_print_timings: prompt eval time =  1322.19 ms /    32 tokens (   41.32 ms per token,    24.20 tokens per second)
llama_print_timings:        eval time = 64654.79 ms /   391 runs   (  165.36 ms per token,     6.05 tokens per second)
llama_print_timings:       total time = 66747.38 ms


"\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The following are common symptoms for appendicitis:\n\n1. Abdominal pain, usually starting around the navel and then shifting to the lower right side of the abdomen.\n2. Loss of appetite.\n3. Nausea and vomiting.\n4. Fever (often low-grade).\n5. Constipation or diarrhea.\n6. Abdominal swelling and rigidity.\n7. Pain upon walking, coughing, or other jarring movements.\n8. Inability to pass gas or have a bowel movement.\n\nAppendicitis cannot be cured via medicine alone due to the risk of appendix rupture, which can lead to peritonitis, a serious and potentially life-threatening condition. If an appendix ruptures, bacteria from the appendix can spill into the abdominal cavity, causing widespread infection. Therefore, when appendicitis is suspected based on symptoms, prompt surgical intervention is necessary.\n\nThe standard surgical procedure for tr

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [21]:
prompt_input3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(prompt_input3, temperature= 0.15, max_tokens=512, top_p=0.94, top_k= 20)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =   175.34 ms /   476 runs   (    0.37 ms per token,  2714.74 tokens per second)
llama_print_timings: prompt eval time =  1415.71 ms /    34 tokens (   41.64 ms per token,    24.02 tokens per second)
llama_print_timings:        eval time = 78620.30 ms /   475 runs   (  165.52 ms per token,     6.04 tokens per second)
llama_print_timings:       total time = 80990.78 ms


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the sudden loss of hair in small, round patches on the scalp or other areas of the body. The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system.\n\nThere is no cure for alopecia areata, but various treatments can help stimulate hair regrowth and potentially prevent further hair loss. Some effective treatments include:\n\n1. Corticosteroids: These medications can be applied directly to the affected area or taken orally to reduce inflammation and suppress the immune system's attack on the hair follicles. Topical corticosteroids are often used for small patches of alopecia areata, while oral corticosteroids may be prescribed for larger areas or more severe cases.\n2. Immunotherapy: Injections of certain substances, such as diphencyprone or squaric acid dibutyl ester, can help stimulate an immune response that promotes hair reg

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [22]:
prompt_input4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(prompt_input4, temperature= 0.20, max_tokens=512, top_p=0.92, top_k= 25)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =   177.86 ms /   478 runs   (    0.37 ms per token,  2687.48 tokens per second)
llama_print_timings: prompt eval time =  1181.31 ms /    28 tokens (   42.19 ms per token,    23.70 tokens per second)
llama_print_timings:        eval time = 78817.04 ms /   477 runs   (  165.23 ms per token,     6.05 tokens per second)
llama_print_timings:       total time = 80972.35 ms


"\n\nI cannot provide personal medical advice or recommendations for specific treatments as I don't have access to your medical history or current condition. However, I can share some general information about the treatment options for brain injuries based on established medical guidelines and reputable health sources such as the World Health Organization (WHO), Centers for Disease Control and Prevention (CDC), and Mayo Clinic.\n\nThe treatment for a brain injury depends on the severity, location, and extent of the damage. Here are some common treatments:\n\n1. Emergency care: For severe brain injuries, emergency care is crucial to prevent further damage or complications. This may include surgery to remove hematomas (clots) or repair skull fractures, as well as measures to maintain adequate blood pressure and oxygen levels.\n2. Medications: Depending on the symptoms, various medications may be prescribed to manage conditions such as seizures, infections, pain, or swelling in the brain.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [23]:
prompt_input5 = system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(prompt_input5, temperature= 0.25, max_tokens=512, top_p=0.90, top_k= 30)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =   155.33 ms /   422 runs   (    0.37 ms per token,  2716.74 tokens per second)
llama_print_timings: prompt eval time =  1465.25 ms /    35 tokens (   41.86 ms per token,    23.89 tokens per second)
llama_print_timings:        eval time = 69529.88 ms /   421 runs   (  165.15 ms per token,     6.05 tokens per second)
llama_print_timings:       total time = 71838.40 ms


"\n\nA leg fracture, specifically a tibia or fibula fracture, is a common injury that can occur during hiking due to falls, tripping over rocks, or other traumatic events. Here are the necessary precautions and treatment steps for such an individual:\n\n1. Assess the severity of the injury: If the person is in significant pain, unable to bear weight on the affected leg, or shows signs of swelling, bruising, or deformity, it's essential to suspect a fracture and seek medical attention immediately.\n\n2. Provide first aid: Apply a sterile dressing to control any bleeding, and use a splint or sling to immobilize the injured leg to prevent further damage and provide comfort. Do not attempt to realign the bone or push it back into place yourself.\n\n3. Transport the person safely: If possible, carry the individual to a vehicle or arrange for transportation to the nearest medical facility. Ensure they are as comfortable as possible during transport and keep them still to minimize discomfort 

## Data Preparation for RAG

### Loading the Data

In [ ]:
medical_diagnosis_manual_path = "/root/ashiravi/medical_diagnosis_manual.pdf"

In [ ]:
pdf_loader = PyMuPDFLoader(medical_diagnosis_manual_path)

In [ ]:
medical_diagnosis_manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(medical_diagnosis_manual[i].page_content,end="\n")

Page Number : 1
ashish.ravindranathan@gmail.com
07QVFMNXP9
or personal use by ashish.ravindranatha
shing the contents in part or full is liable 

Page Number : 2
ashish.ravindranathan@gmail.com
07QVFMNXP9
This file is meant for personal use by ashish.ravindranathan@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................................

#### Checking the number of pages

In [28]:
len(medical_diagnosis_manual)

4114

### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)

In [30]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [31]:
len(document_chunks)

8503

In [32]:
document_chunks[0].page_content

'ashish.ravindranathan@gmail.com\n07QVFMNXP9\nor personal use by ashish.ravindranatha\nshing the contents in part or full is liable'

In [33]:
document_chunks[1].page_content

'ashish.ravindranathan@gmail.com\n07QVFMNXP9\nThis file is meant for personal use by ashish.ravindranathan@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [34]:
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\nashish.ravindranathan@gmail.com\n07QVFMNXP9\nThis file is meant for personal use by ashish.ravindranathan@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [35]:
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

##### Observation

As expected, there are some overlaps  
If we increase the `chunk_overlap`, the overlapping length of the sentence will also increase.

### Embedding

In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [38]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

In [39]:
import chromadb

# Initialize the Chroma client
# For a persistent client:
# client = chromadb.PersistentClient(path="/path/to/your/chroma/db")
# For an in-memory client:
client = chromadb.Client()

# Get the version
version = client.get_version()
print(f"ChromaDB client version: {version}")

ChromaDB client version: 1.1.0


### Vector Database

In [ ]:
out_dir_new = 'medical_db'

if not os.path.exists(out_dir_new):
  os.makedirs(out_dir_new)

In [ ]:
# Define a batch size that is safely below the limit
batch_size = 2000 

# Initialize the vector store with the first batch of documents
print(f"Creating vector store with the first batch of {min(batch_size, len(document_chunks))} documents...")
vectorstore = Chroma.from_documents(
    documents=document_chunks[:batch_size],
    embedding=embedding_model,
    persist_directory=out_dir_new
)

# Loop through the rest of the documents and add them in batches
for i in range(batch_size, len(document_chunks), batch_size):
    # Select the next batch
    batch = document_chunks[i:i + batch_size]
    
    print(f"Adding batch of {len(batch)} documents (from index {i})...")
    
    # Use the add_documents method for subsequent batches
    vectorstore.add_documents(documents=batch)

# Persist the collection to disk once all documents are added
print("All batches have been added. Persisting the database...")
vectorstore.persist()

print("Vector store created and persisted successfully.")

Creating vector store with the first batch of 2000 documents...
Adding batch of 2000 documents (from index 2000)...
Adding batch of 2000 documents (from index 4000)...
Adding batch of 2000 documents (from index 6000)...
Adding batch of 503 documents (from index 8000)...
All batches have been added. Persisting the database...
Vector store created and persisted successfully.


In [ ]:
vectorstore = Chroma(persist_directory=out_dir_new,embedding_function=embedding_model)

In [44]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False)

In [45]:
vectorstore.similarity_search("how to treat appendicitis",k=4)

[Document(page_content="• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can\nusually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these\ncases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to\nappendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis\nand a normal cecum, the appendix should be removed.\nAppendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For\nnonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics\nshould be continued until the patient's temperature and WBC count have normalized or continued for a\nfixed course, according to the surgeon's pr

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)

In [47]:
rel_docs = retriever.get_relevant_documents("how to treat appendicitis")
rel_docs

[Document(page_content="• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can\nusually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these\ncases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to\nappendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis\nand a normal cecum, the appendix should be removed.\nAppendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For\nnonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics\nshould be continued until the patient's temperature and WBC count have normalized or continued for a\nfixed course, according to the surgeon's pr

In [48]:
model_output = llm(
    "prompt",
    max_tokens=128,
    temperature=0,
)


model_output['choices'][0]['text']

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    49.03 ms /   117 runs   (    0.42 ms per token,  2386.49 tokens per second)
llama_print_timings: prompt eval time =     0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time = 19176.23 ms /   117 runs   (  163.90 ms per token,     6.10 tokens per second)
llama_print_timings:       total time = 19384.04 ms


' = "Write a Python function that takes a list of numbers and returns the sum of all even numbers in it."\n\ndef sum_even_numbers(numbers):\n    total = 0\n    for number in numbers:\n        if number % 2 == 0:\n            total += number\n    return total\n\n# Test the function\nnumbers = [1, 2, 3, 4, 5, 6]\nprint(sum_even_numbers(numbers)) # Output: 12'

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
qna_system_message = """
You are a Retrieval-Augmented Generation (RAG) AI assistant specializing in the medical domain. Your purpose is to assist healthcare professionals by providing clear, concise, and evidence-based medical information derived strictly from the medical diagnosis manual provided.
Instructions:  
1. Base all answers only on the medical diagnosis manual provided — do not invent or assume facts.
2. Present information in a structured, concise, and clinically useful manner for quick reference.
3. Avoid offering personal medical advice, diagnoses, or treatment instructions tailored to an individual.
4. Clearly indicate the source or context if possible (e.g., "According to the WHO guidelines...").
5. If the retrieved data is insufficient to answer, state this clearly and suggest consulting a qualified healthcare professional.
"""

In [50]:
qna_user_message_template = """
###Context
Based on the retrieved content from authoritative medical sources, answer the following:
{context}

###Respond to the question:
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [52]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.67 ms /   128 runs   (    0.42 ms per token,  2384.81 tokens per second)
llama_print_timings: prompt eval time = 122729.25 ms /  2836 tokens (   43.28 ms per token,    23.11 tokens per second)
llama_print_timings:        eval time = 22536.14 ms /   127 runs   (  177.45 ms per token,     5.64 tokens per second)
llama_print_timings:       total time = 145503.87 ms


'According to the medical diagnosis manual provided, there is no specific information regarding the protocol for managing sepsis in a critical care unit within this text. However, it does mention that prevention of infection is an essential aspect of supportive care for critically ill patients (p. 3480). Early recognition and prompt treatment of sepsis are crucial to improve outcomes. The manual suggests monitoring vital signs, quantifying fluid intake and output, and performing daily weight checks. If a patient develops suspected sepsis, they should be evaluated by a healthcare professional and treated according to current guidelines, such as the Surviving'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [53]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.74 ms /   128 runs   (    0.42 ms per token,  2382.02 tokens per second)
llama_print_timings: prompt eval time = 120922.08 ms /  2763 tokens (   43.76 ms per token,    22.85 tokens per second)
llama_print_timings:        eval time = 22632.16 ms /   127 runs   (  178.21 ms per token,     5.61 tokens per second)
llama_print_timings:       total time = 143792.11 ms


'The common symptoms for appendicitis include abdominal pain, usually localized in the right lower quadrant of the abdomen, loss of appetite, nausea, vomiting, constipation or diarrhea, fever, and a feeling of ill health. Appendicitis cannot be cured via medicine alone as it requires surgical intervention to remove the inflamed appendix. The treatment for appendicitis is open or laparoscopic appendectomy. This procedure should be preceded by IV antibiotics, usually third-generation cephalosporins, to prevent infection spread'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [54]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    54.03 ms /   128 runs   (    0.42 ms per token,  2369.14 tokens per second)
llama_print_timings: prompt eval time = 87799.92 ms /  2052 tokens (   42.79 ms per token,    23.37 tokens per second)
llama_print_timings:        eval time = 22192.30 ms /   127 runs   (  174.74 ms per token,     5.72 tokens per second)
llama_print_timings:       total time = 110230.03 ms


'According to the provided text from The Merck Manual of Diagnosis & Therapy, 19th Edition, the condition described as sudden patchy hair loss is known as alopecia areata. This condition is characterized by the sudden appearance of bald spots on the scalp or any other hair-bearing areas. Alopecia areata is considered an autoimmune disorder that affects genetically susceptible individuals exposed to unclear environmental triggers.\n\nThe text suggests several treatment options for alopecia areata, including:\n1. Topical corticosteroids: These'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [55]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.78 ms /   128 runs   (    0.42 ms per token,  2379.98 tokens per second)
llama_print_timings: prompt eval time = 113283.25 ms /  2602 tokens (   43.54 ms per token,    22.97 tokens per second)
llama_print_timings:        eval time = 22486.83 ms /   127 runs   (  177.06 ms per token,     5.65 tokens per second)
llama_print_timings:       total time = 136010.45 ms


'According to the medical diagnosis manual provided, initial treatment for a person with a traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabil'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [56]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.73 ms /   128 runs   (    0.42 ms per token,  2382.06 tokens per second)
llama_print_timings: prompt eval time = 92107.68 ms /  2149 tokens (   42.86 ms per token,    23.33 tokens per second)
llama_print_timings:        eval time = 22331.15 ms /   127 runs   (  175.84 ms per token,     5.69 tokens per second)
llama_print_timings:       total time = 114677.73 ms


'According to The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 323. Fractures, Dislocations & Sprains:\n\nFor a person who has fractured their leg during a hiking trip, the following precautions and treatment steps should be taken:\n\n1. Immobilize the injured limb using elastic bandages (layers 2 and 4) for the first 2 days to minimize swelling. Elevate the limb above the heart level during this period. After 48 hours, apply periodic warmth using'

### Fine-tuning parameters

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [57]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1, max_tokens=256)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    82.11 ms /   195 runs   (    0.42 ms per token,  2374.89 tokens per second)
llama_print_timings: prompt eval time = 115034.79 ms /  2639 tokens (   43.59 ms per token,    22.94 tokens per second)
llama_print_timings:        eval time = 34534.66 ms /   194 runs   (  178.01 ms per token,     5.62 tokens per second)
llama_print_timings:       total time = 149941.75 ms


"According to the information provided in the medical diagnosis manual, there is no specific protocol mentioned for managing sepsis in a critical care unit within this text. However, it does mention that prevention of infection is an essential part of supportive care for ICU patients (see p. 3480). Early recognition and prompt treatment of sepsis are crucial to improve outcomes. If a patient is suspected to have sepsis, healthcare professionals should follow established sepsis guidelines, such as the Surviving Sepsis Campaign: Guidelines for Management of Critically Ill Adults (SSC) or other relevant clinical practice guidelines. These guidelines provide evidence-based recommendations on diagnosis, treatment, and management of sepsis in critical care settings. It is important to note that individual patient circumstances may require adjustments to the standard protocol based on their specific condition. Always consult a qualified healthcare professional for medical advice tailored to a

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [58]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,k=2,top_k=25)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    55.66 ms /   128 runs   (    0.43 ms per token,  2299.51 tokens per second)
llama_print_timings: prompt eval time = 121073.70 ms /  2763 tokens (   43.82 ms per token,    22.82 tokens per second)
llama_print_timings:        eval time = 22655.97 ms /   127 runs   (  178.39 ms per token,     5.61 tokens per second)
llama_print_timings:       total time = 143980.12 ms


'The common symptoms for appendicitis include abdominal pain, usually localized in the right lower quadrant of the abdomen, loss of appetite, nausea, vomiting, constipation or diarrhea, fever, and a feeling of ill health. Appendicitis cannot be cured via medicine alone as it requires surgical intervention to remove the inflamed appendix. The treatment for appendicitis is open or laparoscopic appendectomy. This procedure should be preceded by IV antibiotics, usually third-generation cephalosporins, to prevent infection spread'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [59]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,k=5,max_tokens=256,temperature=0,top_k=10)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =   107.36 ms /   256 runs   (    0.42 ms per token,  2384.55 tokens per second)
llama_print_timings: prompt eval time = 87849.19 ms /  2052 tokens (   42.81 ms per token,    23.36 tokens per second)
llama_print_timings:        eval time = 44722.55 ms /   255 runs   (  175.38 ms per token,     5.70 tokens per second)
llama_print_timings:       total time = 133079.47 ms


'According to the provided text from The Merck Manual of Diagnosis & Therapy, 19th Edition, the condition described as sudden patchy hair loss is known as alopecia areata. This condition is characterized by the sudden appearance of bald spots on the scalp or any other hair-bearing areas. Alopecia areata is considered an autoimmune disorder that affects genetically susceptible individuals exposed to unclear environmental triggers.\n\nThe text suggests several treatment options for alopecia areata, including:\n1. Topical corticosteroids: These medications can stimulate hair growth and are usually effective within 6 to 8 months of treatment. However, they may have adverse effects such as decreased libido, erectile dysfunction, hypersensitivity reactions, gynecomastia, and myopathy.\n2. Topical minoxidil: This medication can also stimulate hair growth but is less effective than corticosteroids. It may take up to 12 weeks to see results.\n3. Topical anthralin: This medication is used for se

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [60]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,k=3,max_tokens=128,temperature=0,top_p=0.99,top_k=30)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.83 ms /   128 runs   (    0.42 ms per token,  2377.94 tokens per second)
llama_print_timings: prompt eval time = 113455.88 ms /  2602 tokens (   43.60 ms per token,    22.93 tokens per second)
llama_print_timings:        eval time = 22545.21 ms /   127 runs   (  177.52 ms per token,     5.63 tokens per second)
llama_print_timings:       total time = 136251.01 ms


'According to the medical diagnosis manual provided, initial treatment for a person with a traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabil'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [61]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,k=5,temperature=0,top_p=0.98,top_k=20)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    53.91 ms /   128 runs   (    0.42 ms per token,  2374.11 tokens per second)
llama_print_timings: prompt eval time = 92121.15 ms /  2149 tokens (   42.87 ms per token,    23.33 tokens per second)
llama_print_timings:        eval time = 22294.92 ms /   127 runs   (  175.55 ms per token,     5.70 tokens per second)
llama_print_timings:       total time = 114664.44 ms


'According to The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 323. Fractures, Dislocations & Sprains:\n\nFor a person who has fractured their leg during a hiking trip, the following precautions and treatment steps should be taken:\n\n1. Immobilize the injured limb using elastic bandages (layers 2 and 4) for the first 2 days to minimize swelling. Elevate the limb above the heart level during this period. After 48 hours, apply periodic warmth using'

## Output Evaluation

In [ ]:
groundedness_rater_system_message  = """
You are a groundedness evaluator.
Review the AI’s response to determine if it is factually correct and fully supported by information from the provided trusted medical source — the Merck Manuals, defined in the Data Dictionary.
Assess whether all parts of the answer can be substantiated with explicit references from the retrieved source content. Avoid introducing any external or speculative information.
In your evaluation, highlight:
    Pass ✅ if the answer is entirely based on provided/retrieved source data.
    Fail ❌ if any part of the answer is unsupported by the source data or contradicts it.
    Only judge groundedness — do not assess relevance, style, or completeness unless it affects factual support.
"""

In [63]:
relevance_rater_system_message = """
You are a relevance evaluator.
Determine whether the AI’s answer fully addresses the user’s question in the healthcare RAG business context provided.
Focus on:
    Whether the answer stays on topic and directly answers the question asked.
    Whether the information is appropriate for healthcare professionals (without giving personal medical advice).
    Whether it avoids unnecessary tangents or unrelated content.
Mark:
    Pass ✅ if the response is completely relevant and aligned with the stated question and context.
    Fail ❌ if the response contains irrelevant, off-topic, or extraneous information.
    Do not evaluate groundedness or factual accuracy; only judge topical and contextual relevance.
"""

In [ ]:
user_message_template = """ 
User Query: {question}
Context: {context}
Response: {answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [73]:
grounded,relevance = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?",max_tokens=400)

print(grounded,end="\n\n")
print(relevance)

Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    52.90 ms /   122 runs   (    0.43 ms per token,  2306.19 tokens per second)
llama_print_timings: prompt eval time = 123599.94 ms /  2844 tokens (   43.46 ms per token,    23.01 tokens per second)
llama_print_timings:        eval time = 21451.91 ms /   121 runs   (  177.29 ms per token,     5.64 tokens per second)
llama_print_timings:       total time = 145299.66 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    41.54 ms /    99 runs   (    0.42 ms per token,  2383.36 tokens per second)
llama_print_timings: prompt eval time = 127200.91 ms /  2930 tokens (   43.41 ms per token,    23.03 tokens per second)
llama_print_timings:        eval time = 17407.73 ms /    98 runs   (  177.63 ms per token,     5.63 tokens per second)
llama_print_timings:       total time = 144804.14 ms
Llama.

 Fail ❌: The Merck Manuals do not provide specific information on the protocol for managing sepsis in a critical care unit beyond mentioning the importance of preventing infection. The response goes beyond the provided source data by suggesting that suspected sepsis patients should be evaluated and treated with antibiotics, fluid resuscitation, and other measures as needed. While these are common steps in sepsis management, they were not explicitly stated or supported by the Merck Manuals.

 Fail ❌

The response does not fully address the user's question about the protocol for managing sepsis in a critical care unit. While it mentions that prevention of infection is part of supportive care for ICU patients, it does not provide specific information on the steps or procedures involved in managing sepsis once it has been diagnosed. The response also includes some tangential information about preventing infection and the role of infectious disease specialists.



llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    38.56 ms /    92 runs   (    0.42 ms per token,  2385.77 tokens per second)
llama_print_timings: prompt eval time = 126938.40 ms /  2923 tokens (   43.43 ms per token,    23.03 tokens per second)
llama_print_timings:        eval time = 16298.87 ms /    91 runs   (  179.11 ms per token,     5.58 tokens per second)
llama_print_timings:       total time = 143421.16 ms


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [74]:
grounded,relevance = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",max_tokens=400)

print(grounded,end="\n\n")
print(relevance)

Llama.generate: prefix-match hit



llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    81.77 ms /   195 runs   (    0.42 ms per token,  2384.83 tokens per second)
llama_print_timings: prompt eval time = 129037.47 ms /  2968 tokens (   43.48 ms per token,    23.00 tokens per second)
llama_print_timings:        eval time = 34575.15 ms /   194 runs   (  178.22 ms per token,     5.61 tokens per second)
llama_print_timings:       total time = 164003.06 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    30.65 ms /    73 runs   (    0.42 ms per token,  2381.81 tokens per second)
llama_print_timings: prompt eval time = 136742.70 ms /  3126 tokens (   43.74 ms per token,    22.86 tokens per second)
llama_print_timings:        eval time = 12831.84 ms /    72 runs   (  178.22 ms per token,     5.61 tokens per second)
llama_print_timings:       total time = 149721.35 ms
Llama.generate: prefix-match hit


 Pass ✅

The user's query and the provided context were both consistent with the information from the Merck Manuals regarding symptoms of appendicitis, the need for surgical intervention (appendectomy), and the use of antibiotics before and after surgery. The response accurately summarized this information without introducing any external or speculative elements.

 Pass ✅

The AI's answer directly addresses the user's question about the common symptoms of appendicitis and whether it can be cured via medicine or if surgical intervention is required. The information provided is appropriate for healthcare professionals as it focuses on medical treatments and procedures related to appendicitis without giving personal medical advice. The response avoids unnecessary tangents or unrelated content, staying strictly on topic.



llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    38.29 ms /    87 runs   (    0.44 ms per token,  2272.31 tokens per second)
llama_print_timings: prompt eval time = 136429.26 ms /  3119 tokens (   43.74 ms per token,    22.86 tokens per second)
llama_print_timings:        eval time = 15342.51 ms /    86 runs   (  178.40 ms per token,     5.61 tokens per second)
llama_print_timings:       total time = 151952.46 ms


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [75]:
grounded,relevance = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",max_tokens=400)

print(grounded,end="\n\n")
print(relevance)

Llama.generate: prefix-match hit



llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =   169.54 ms /   400 runs   (    0.42 ms per token,  2359.37 tokens per second)
llama_print_timings: prompt eval time = 95650.71 ms /  2257 tokens (   42.38 ms per token,    23.60 tokens per second)
llama_print_timings:        eval time = 70085.91 ms /   399 runs   (  175.65 ms per token,     5.69 tokens per second)
llama_print_timings:       total time = 166574.40 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    32.66 ms /    78 runs   (    0.42 ms per token,  2388.24 tokens per second)
llama_print_timings: prompt eval time = 112600.25 ms /  2620 tokens (   42.98 ms per token,    23.27 tokens per second)
llama_print_timings:        eval time = 13631.31 ms /    77 runs   (  177.03 ms per token,     5.65 tokens per second)
llama_print_timings:       total time = 126385.95 ms
Llama.generate: prefix-match hit


 Pass ✅

The response accurately summarizes the information from The Merck Manual of Diagnosis & Therapy regarding alopecia areata and its treatments. The provided source data mentions corticosteroids, retinoids, and immunosuppressants as potential treatment options for alopecia areata, which is reflected in the response.

 Pass ✅

The AI's answer directly addresses the user's question about effective treatments for sudden patchy hair loss (alopecia areata) and possible causes. The information provided is relevant to healthcare professionals and stays on topic without any unnecessary tangents or unrelated content.



llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    25.21 ms /    60 runs   (    0.42 ms per token,  2380.10 tokens per second)
llama_print_timings: prompt eval time = 112460.24 ms /  2613 tokens (   43.04 ms per token,    23.23 tokens per second)
llama_print_timings:        eval time = 10420.38 ms /    59 runs   (  176.62 ms per token,     5.66 tokens per second)
llama_print_timings:       total time = 123000.97 ms


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [76]:
grounded,relevance = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",max_tokens=400)

print(grounded,end="\n\n")
print(relevance)

Llama.generate: prefix-match hit



llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    63.78 ms /   152 runs   (    0.42 ms per token,  2383.04 tokens per second)
llama_print_timings: prompt eval time = 121443.80 ms /  2807 tokens (   43.26 ms per token,    23.11 tokens per second)
llama_print_timings:        eval time = 26787.24 ms /   151 runs   (  177.40 ms per token,     5.64 tokens per second)
llama_print_timings:       total time = 148536.76 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    49.83 ms /   114 runs   (    0.44 ms per token,  2287.92 tokens per second)
llama_print_timings: prompt eval time = 126995.36 ms /  2921 tokens (   43.48 ms per token,    23.00 tokens per second)
llama_print_timings:        eval time = 20088.87 ms /   113 runs   (  177.78 ms per token,     5.63 tokens per second)
llama_print_timings:       total time = 147312.92 ms
Llama.generate: prefix-match hit


 Pass ✅

The AI's response is entirely based on the provided source data from the Merck Manuals. It accurately summarizes the initial treatment for a person with traumatic brain injury (TBI), including ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, and the potential need for surgery to treat intracranial pressure or hematomas. The response also mentions the importance of rehabilitation for many patients. There are no unsupported statements or contradictions in the AI's answer.

 Pass ✅

The AI's answer directly addresses the user's question about treatments for a person with brain tissue injury by discussing the initial medical interventions and subsequent rehabilitation, which is in line with the context provided in the text. The information given is appropriate for healthcare professionals as it focuses on medical treatments and does not provide personal medical advice or irrelevant content.



llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    32.44 ms /    77 runs   (    0.42 ms per token,  2373.83 tokens per second)
llama_print_timings: prompt eval time = 126740.28 ms /  2914 tokens (   43.49 ms per token,    22.99 tokens per second)
llama_print_timings:        eval time = 13508.86 ms /    76 runs   (  177.75 ms per token,     5.63 tokens per second)
llama_print_timings:       total time = 140403.51 ms


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [77]:
grounded,relevance = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",max_tokens=400)

print(grounded,end="\n\n")
print(relevance)

Llama.generate: prefix-match hit



llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =   170.10 ms /   400 runs   (    0.43 ms per token,  2351.50 tokens per second)
llama_print_timings: prompt eval time = 100401.71 ms /  2354 tokens (   42.65 ms per token,    23.45 tokens per second)
llama_print_timings:        eval time = 70369.84 ms /   399 runs   (  176.37 ms per token,     5.67 tokens per second)
llama_print_timings:       total time = 171621.41 ms
Llama.generate: prefix-match hit

llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    19.32 ms /    46 runs   (    0.42 ms per token,  2380.58 tokens per second)
llama_print_timings: prompt eval time = 117240.22 ms /  2718 tokens (   43.13 ms per token,    23.18 tokens per second)
llama_print_timings:        eval time =  7974.39 ms /    45 runs   (  177.21 ms per token,     5.64 tokens per second)
llama_print_timings:       total time = 125307.87 ms
Llama.generate: prefix-match hit


 Pass ✅

The response is entirely based on the provided source data from The Merck Manual of Diagnosis & Therapy, 19th Edition. No external or speculative information has been introduced.

 Pass ✅

The AI's answer fully addresses the user's question by providing necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip. The information is directly related to the context provided, which discusses various aspects of treating fractures, including immobilization techniques, wound care, and potential complications. The answer avoids unnecessary tangents or unrelated content and stays on topic throughout.



llama_print_timings:        load time =   805.69 ms
llama_print_timings:      sample time =    39.11 ms /    93 runs   (    0.42 ms per token,  2377.79 tokens per second)
llama_print_timings: prompt eval time = 116951.11 ms /  2711 tokens (   43.14 ms per token,    23.18 tokens per second)
llama_print_timings:        eval time = 16245.36 ms /    92 runs   (  176.58 ms per token,     5.66 tokens per second)
llama_print_timings:       total time = 133383.97 ms


## Actionable Insights and Business Recommendations

Actionable Insights

1. RAG Effectively Grounds Most Queries, But Not All: The prototype successfully demonstrates that the RAG approach can retrieve relevant information from the Merck Manuals and generate grounded responses for queries like appendicitis, hair loss, brain injury, and leg fractures, as evidenced by the "Pass ✅" from the groundedness rater. This confirms the core value proposition of using RAG for medical information retrieval.
2. Gaps in Knowledge Base Lead to Groundedness Failures: A critical insight is the "Fail ❌" for the sepsis management protocol query. This highlights that while the LLM has general medical knowledge, it will "hallucinate" or infer beyond the provided source if the specific answer isn't present, even with a system prompt aiming for strict adherence. This is a significant risk for medical applications.
3. LLM's Tendency to "Be Helpful" Can Override Strict Groundedness: The prototype shows that the underlying LLM's propensity to generate a coherent, helpful response can sometimes override the strict instruction to only use the provided context or state when information is unavailable. This is a key challenge for maintaining factual accuracy in a medical RAG system.
4. Evaluation Framework is Essential for Quality Control: The inclusion of both groundedness and relevance raters (even if LLM-based) within the prototype is crucial. The explicit "Fail ❌" for the sepsis query demonstrates the value of this framework in identifying instances where the RAG system does not perform as intended, preventing potentially misleading or unverified information from being presented.



Business Recommendations 

1. Strategically Expand and Curate the Knowledge Base:

Action: Analyze queries that result in "Fail ❌" for groundedness (like the sepsis protocol). If these are critical clinical topics, actively seek to integrate additional, authoritative medical guidelines (e.g., Surviving Sepsis Campaign, specific critical care society protocols) into the RAG system to fill these identified knowledge gaps.

Business Impact: Ensures comprehensive coverage for high-priority medical queries, increasing the utility and reliability of the system for healthcare professionals. Avoids frustrating users with "no information" responses for common or critical conditions.

2. Optimize Performance for Production Deployment:

Action: Explore techniques to reduce inference latency, such as using more powerful GPUs, optimizing LLM quantization, or investigating advanced RAG techniques like "query rewriting" or "context summarization" to reduce the size of the prompt fed to the LLM.

Business Impact: Ensures the system is responsive enough for real-time clinical decision-making, improving user experience and adoption in fast-paced healthcare environments.

3. Develop a "Source Traceability" Feature:

Action: For every generated response, provide clear references (e.g., page numbers, chapter titles) from the Merck Manuals or other integrated sources that support the answer.

Business Impact: Empowers healthcare professionals to quickly verify information, builds transparency and trust in the AI, and supports evidence-based practice by allowing users to delve deeper into the original source if needed.